# 🎯 Centralized Deep Learning Training Pipeline (Scenario E1)
## Intrusion Detection on CICIoT2023 with TabularIoTMLP & Multi-Class Focal Loss

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NVKQ2022/FedLearning/blob/main/notebooks/centralized_training_pipeline.ipynb)

This notebook provides the complete, publication-grade **Centralized Training Pipeline (Scenario E1)** for the **FL-IoT-IDS** project. It trains the lightweight `TabularIoTMLP` neural network on centralized network flow data to establish the **empirical performance upper bound ceiling** against which all distributed Federated Learning simulations are benchmarked.

### Pipeline Architecture:
1. **Environment Setup & Synchronization:** Git pull repository and install dependencies.
2. **🌟 Unified Hyperparameter Configuration:** Single declarative `ExperimentConfig` defining all data, architecture, loss, optimizer, and checkpointing settings.
3. **Persistent Dataset Mounting:** Load the real 991 MB `merged_CICIOT2023_data.csv` directly from Google Drive.
4. **Leak-Free Tabular Preprocessing:** Stratified train/val/test split (70/10/20), $\pm\infty$ cleaning, training median imputation, non-negative physical clipping, $\log(1+x)$ heavy-tail compression, and `RobustScaler` IQR normalization.
5. **Class Imbalance Strategy:** Configurable class weighting (`balanced`, `sqrt_balanced`, or `none`) with `MultiClassFocalLoss` or standard Cross-Entropy.
6. **Training Execution:** `CentralizedTrainer` with AdamW, gradient clipping, live `tqdm` progress bar, validation tracking, and configurable checkpointing (`monitor='val_loss'` or `'val_acc'`).
7. **Rigorous Evaluation:** Comprehensive holdout evaluation reporting Macro-F1, isolated Minority Recall (`Web-based`, `Brute-force`), and row-normalized Confusion Matrix.
8. **Artifact Archival:** Automatically saves model checkpoints, preprocessor state, and hyperparameter config to Google Drive.

---
### 🛠️ Step 1: Environment Setup & Repository Synchronization
We check GPU hardware acceleration, pull the latest code from `https://github.com/NVKQ2022/FedLearning.git`, and install dependencies.

In [ ]:
# 1. Hardware acceleration verification
import torch
print(f"PyTorch Version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")


In [ ]:
# 2. Clone or pull the latest repository codebase
import os, sys, subprocess

repo_url = "https://github.com/NVKQ2022/FedLearning.git"
repo_dir = "/content/FedLearning"

if not os.path.exists(repo_dir):
    print(f"Cloning repository from {repo_url}...")
    subprocess.run(["git", "clone", repo_url, repo_dir], check=True)
else:
    print("Repository already present. Pulling latest updates from origin/main...")
    subprocess.run(["git", "-C", repo_dir, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", repo_dir, "reset", "--hard", "origin/main"], check=True)

os.chdir(repo_dir)
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

# Purge any cached in-memory src modules so fresh code is loaded
for mod in list(sys.modules.keys()):
    if mod.startswith("src"):
        del sys.modules[mod]

print(f"✅ Working directory set to: {os.getcwd()}")


In [ ]:
# 3. Install dependencies from requirements.txt
!pip install -q --no-warn-conflicts -r requirements.txt
print("✅ Dependencies verified and installed.")


---
### ⚙️ Step 2: Global Hyperparameter Configuration & Reproducibility Seed
**Edit any hyperparameters here before running the pipeline.** All subsequent cells consume settings directly from this single source of truth.

In [ ]:
from src.utils.config import ExperimentConfig
from src.utils.seed import set_seed

# ==============================================================================
# 🛠️ EDIT HYPERPARAMETERS HERE (Single Source of Truth)
# ==============================================================================
CONFIG = ExperimentConfig(
    experiment_name="centralized_e1_baseline",
    seed=42,                            # Integer seed for 100% bit-level reproducibility
    
    # --- Dataset & Sampling ---
    sample_size=100000,                 # 100000 for rapid iteration, or None for full 5.11M rows
    test_size=0.2,                      # 20% holdout test set
    val_size=0.1,                       # 10% validation set
    scaler_type="robust",               # 'robust' (median/IQR) or 'standard'
    batch_size=128,                     # Mini-batch size
    
    # --- Model Architecture ---
    hidden_dims=(128, 64),              # Default (128, 64) or deeper funnel (256, 128, 64)
    dropout_rate=0.2,                   # Dropout probability
    
    # --- Loss & Imbalance Strategy ---
    loss_type="focal_loss",             # 'focal_loss', 'cross_entropy', or 'weighted_ce'
    class_weight_strategy="balanced",   # 'balanced' (inverse frequency), 'sqrt_balanced', or 'none'
    focal_gamma=2.0,                    # Focusing parameter (higher = more focus on rare attacks)
    
    # --- Optimization & Training ---
    optimizer_type="adamw",             # 'adamw', 'adam', or 'sgd'
    learning_rate=1e-3,
    weight_decay=1e-4,
    epochs=15,                          # Maximum training epochs
    patience=4,                         # Early stopping patience
    monitor_metric="val_loss",          # 'val_loss' (lowest loss) or 'val_acc' (highest accuracy)
    max_grad_norm=5.0,
    verbose=True
)

# Enforce global deterministic seed across Python, NumPy, and PyTorch (CPU & CUDA)
set_seed(CONFIG.seed)

# Print formatted hyperparameter overview
CONFIG.display()


---
### 💾 Step 3: Google Drive Mounting & Dataset Acquisition
We mount Google Drive to load the real **991.00 MB** dataset (`merged_CICIOT2023_data.csv`).

* **Target Path in Google Drive:** `/content/drive/MyDrive/NguyenVietKyQuanKLTN/datasets/merged_CICIOT2023_data.csv`
* **Local SSD Fast Cache:** `/content/FedLearning/datasets/CICIOT2023/merged_CICIOT2023_data.csv`

In [ ]:
# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
except Exception as e:
    print('Drive mount skipped or not running on Colab:', e)

import os, shutil

target_csv_path = CONFIG.target_csv_path
os.makedirs(os.path.dirname(target_csv_path), exist_ok=True)

gdrive_csv = CONFIG.gdrive_csv
if not os.path.exists(gdrive_csv):
    if os.path.exists(CONFIG.alt_gdrive_csv):
        gdrive_csv = CONFIG.alt_gdrive_csv

if os.path.exists(gdrive_csv):
    gdrive_size = os.path.getsize(gdrive_csv) / (1024**2)
    if not os.path.exists(target_csv_path) or os.path.getsize(target_csv_path) < 100 * 1024 * 1024:
        print(f"Copying real 991 MB dataset from Google Drive to local SSD ({gdrive_size:.2f} MB)...")
        shutil.copy(gdrive_csv, target_csv_path)
    print(f"✅ REAL CICIoT2023 dataset ready on local SSD: {os.path.getsize(target_csv_path) / (1024**2):.2f} MB")
elif os.path.exists(target_csv_path) and os.path.getsize(target_csv_path) >= 100 * 1024 * 1024:
    print(f"✅ REAL CICIoT2023 dataset already present on local SSD: {os.path.getsize(target_csv_path) / (1024**2):.2f} MB")
else:
    print(f"⚠️ Real dataset NOT found at '{gdrive_csv}'.")
    print("Please verify that merged_CICIOT2023_data.csv exists under NguyenVietKyQuanKLTN/datasets/ in Google Drive!")


---
### 🔄 Step 4: Leak-Free Preprocessing & Class Weights
We use `load_and_preprocess_ciciot2023` to perform:
* **Stratified train/val/test split** strictly before computing statistics.
* **Outlier & skew mitigation:** `RobustScaler` (median & IQR) and $\log(1+x)$ compression.
* **Class weight calculation:** Configurable strategy (`CONFIG.class_weight_strategy`) to balance minority attack penalization.

In [ ]:
from src.data.preprocess import load_and_preprocess_ciciot2023, compute_balanced_class_weights
import numpy as np

print(f"Loading and preprocessing CICIoT2023 (sample_size={CONFIG.sample_size}, scaler={CONFIG.scaler_type})...")
data = load_and_preprocess_ciciot2023(
    csv_path=target_csv_path,
    test_size=CONFIG.test_size,
    val_size=CONFIG.val_size,
    sample_size=CONFIG.sample_size,
    scaler_type=CONFIG.scaler_type,
    batch_size=CONFIG.batch_size,
    random_state=CONFIG.seed
)

class_weights = compute_balanced_class_weights(
    y=data["y_train"],
    num_classes=data["num_classes"],
    strategy=CONFIG.class_weight_strategy
)
class_names = data["class_names"]
minority_classes = ["Web-based", "Brute-force"]

print("\n--- Dataset Partition Summary ---")
print(f"Training set:   {len(data['X_train']):,} samples ({len(data['train_loader']):,} batches)")
print(f"Validation set: {len(data['X_val']):,} samples ({len(data['val_loader']):,} batches)")
print(f"Holdout Test:   {len(data['X_test']):,} samples ({len(data['test_loader']):,} batches)")
print("Detected Classes:", class_names)
print(f"Class Weights ({CONFIG.class_weight_strategy}):", np.round(class_weights, 3))


---
### 🧠 Step 5: Model Architecture & Optimization Suite
We instantiate:
* **`TabularIoTMLP`:** Built with `CONFIG.hidden_dims` and `CONFIG.dropout_rate`.
* **Loss Function:** `CONFIG.loss_type` with computed class weights.
* **Optimizer:** `CONFIG.optimizer_type` with decoupled weight decay.

In [ ]:
from src.models.mlp import TabularIoTMLP
from src.losses.focal_loss import build_loss_function
from src.optimizers.optimizer import build_optimizer

# 1. Initialize Neural Network Backbone
model = TabularIoTMLP(
    input_dim=data["input_dim"],
    hidden_dims=CONFIG.hidden_dims,
    num_classes=data["num_classes"],
    dropout_rate=CONFIG.dropout_rate
).to(device)

print(f"Model Trainable Parameters: {model.get_num_parameters():,}")
print(f"Model Weight Payload Size:  {model.get_model_size_kb():.2f} KB")

# 2. Build Loss Function
criterion = build_loss_function(
    loss_type=CONFIG.loss_type,
    class_weights=class_weights if CONFIG.class_weight_strategy != "none" else None,
    gamma=CONFIG.focal_gamma,
    device=device
)

# 3. Build Optimizer
optimizer = build_optimizer(
    model=model,
    optimizer_type=CONFIG.optimizer_type,
    lr=CONFIG.learning_rate,
    weight_decay=CONFIG.weight_decay
)


---
### 🏋️ Step 6: Centralized Model Training with Configurable Monitoring
We use `CentralizedTrainer` with gradient clipping, live `tqdm` mini-batch progress bars, and patience-based validation checkpointing configured to monitor `CONFIG.monitor_metric` (`'val_loss'` or `'val_acc'`).

In [ ]:
from src.training.trainer import CentralizedTrainer

trainer = CentralizedTrainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    max_grad_norm=CONFIG.max_grad_norm
)

print(f"Starting Centralized Training (Epochs={CONFIG.epochs}, Patience={CONFIG.patience}, Monitor='{CONFIG.monitor_metric}', Top-K={CONFIG.save_top_k})...")
history = trainer.fit(
    train_loader=data["train_loader"],
    val_loader=data["val_loader"],
    epochs=CONFIG.epochs,
    patience=CONFIG.patience,
    monitor=CONFIG.monitor_metric,
    save_top_k=CONFIG.save_top_k,
    verbose=CONFIG.verbose
)

best_score = trainer.best_val_loss if CONFIG.monitor_metric == 'val_loss' else trainer.best_val_acc * 100
unit = '' if CONFIG.monitor_metric == 'val_loss' else '%'
print(f"\n✅ Training complete! Best validation {CONFIG.monitor_metric}: {best_score:.4f}{unit}")

print(f"\n🏆 Top-{CONFIG.save_top_k} Best Model Versions Preserved in Memory:")
for item in trainer.get_top_k_summary():
    score_val = f"{item['score']:.4f}" if CONFIG.monitor_metric == 'val_loss' else f"{item['score']*100:.2f}%"
    print(f"   • Rank {item['rank']}: Epoch {item['epoch']:02d} | Val Loss: {item['val_loss']:.4f} | Val Acc: {item['val_acc']*100:.2f}% | Score: {score_val}")


---
### 📈 Step 7: Training Learning Curves
We visualize loss convergence and accuracy progression across training epochs using `src.visualization`.

In [ ]:
from src.visualization import plot_learning_curves

# Plot publication-grade dual-panel loss & accuracy curves
fig_curves = plot_learning_curves(
    history=history,
    title=f"Centralized Training & Validation Convergence ({CONFIG.experiment_name})",
    loss_name=f"{CONFIG.loss_type.capitalize()} Loss",
    save_path="reports/figures/centralized_learning_curves.png",
    show=True
)


---
### 🏆 Step 8: Comprehensive Holdout Test Evaluation
We evaluate the restored best model on the completely unseen holdout test set (20% of data, zero leakage).
We inspect:
* **Overall Accuracy** & **Macro-F1 Score**
* **Per-Class Precision, Recall, and F1**
* **Minority Class Recall Isolation** (`Web-based`, `Brute-force`)
* **Row-Normalized Confusion Matrix Heatmap**

In [ ]:
from src.evaluation.evaluator import evaluate_comprehensive

test_results = evaluate_comprehensive(
    model=model,
    dataloader=data["test_loader"],
    class_names=class_names,
    minority_classes=minority_classes,
    device=device
)

print("=" * 60)
print("🏆 SCENARIO E1 (CENTRALIZED UPPER BOUND) FINAL RESULTS")
print("=" * 60)
print(f"Test Accuracy:    {test_results['accuracy'] * 100:.2f}%")
print(f"Macro-F1 Score:   {test_results['macro_f1'] * 100:.2f}%")
print(f"Weighted-F1:      {test_results['weighted_f1'] * 100:.2f}%")
print(f"Macro-Precision:  {test_results['macro_precision'] * 100:.2f}%")
print(f"Macro-Recall:     {test_results['macro_recall'] * 100:.2f}%")
print("-" * 60)
print("🎯 Minority Class Recall (Stealth Penetration Attacks):")
for cls, rec in test_results["minority_recall"].items():
    print(f"   • {cls:<15}: {rec * 100:.2f}%")
print("-" * 60)
print("📊 Per-Class Breakdown:")
for cname in class_names:
    f1 = test_results["per_class_f1"][cname] * 100
    rec = test_results["per_class_recall"][cname] * 100
    prec = test_results["per_class_precision"][cname] * 100
    supp = test_results["support"][cname]
    print(f"   {cname:<12} | F1: {f1:5.1f}% | Recall: {rec:5.1f}% | Prec: {prec:5.1f}% | Support: {supp:,}")
print("=" * 60)


In [ ]:
from src.visualization import plot_confusion_matrix, plot_per_class_metrics, plot_minority_recall

# 1. Plot Row-Normalized Confusion Matrix Heatmap
fig_cm = plot_confusion_matrix(
    cm=test_results["confusion_matrix"],
    class_names=class_names,
    title=f"Scenario E1: Normalized Confusion Matrix ({CONFIG.experiment_name})",
    save_path="reports/figures/centralized_confusion_matrix.png",
    show=True
)

# 2. Plot Per-Class Precision, Recall, and F1 Breakdown
fig_metrics = plot_per_class_metrics(
    metrics_dict=test_results,
    class_names=class_names,
    title=f"Scenario E1: Per-Class Precision, Recall & F1 ({CONFIG.experiment_name})",
    save_path="reports/figures/centralized_per_class_metrics.png",
    show=True
)

# 3. Plot Stealth Attack Minority Recall Isolation
fig_minority = plot_minority_recall(
    minority_data=test_results["minority_recall"],
    title=f"Scenario E1: Minority Attack Recall Isolation ({CONFIG.experiment_name})",
    save_path="reports/figures/centralized_minority_recall.png",
    show=True
)


---
### 💾 Step 9: Export Checkpoints, Config & Reports to Google Drive
We serialize the best model weights (`.pth`), preprocessor state (`.pkl`), the complete hyperparameter config (`.json`), and evaluation metrics directly to `/content/drive/MyDrive/NguyenVietKyQuanKLTN/` so your findings are permanently archived.

In [ ]:
import json

# 1. Save all Top-K model checkpoints to local checkpoints/ directory
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("reports", exist_ok=True)

saved_ckpt_paths = trainer.save_top_k(
    output_dir="checkpoints",
    prefix=CONFIG.experiment_name,
    monitor=CONFIG.monitor_metric
)

preprocessor_save_path = "checkpoints/preprocessor.pkl"
data["preprocessor"].save(preprocessor_save_path)

config_save_path = "checkpoints/experiment_config.json"
CONFIG.save_json(config_save_path)

metrics_export = {k: v for k, v in test_results.items() if k != "confusion_matrix"}
metrics_file = "reports/centralized_e1_metrics.json"
with open(metrics_file, "w") as f:
    json.dump(metrics_export, f, indent=4)

print(f"✅ Successfully preserved {len(saved_ckpt_paths)} checkpoint artifacts in 'checkpoints/':")
for p in saved_ckpt_paths:
    print(f"   • {p}")
print(f"✅ Preprocessor saved to:   {preprocessor_save_path}")
print(f"✅ Config saved to:         {config_save_path}")
print(f"✅ Metrics report saved:   {metrics_file}")

# 2. Archive all Top-K checkpoints, configs, and reports to Google Drive
gdrive_dest = CONFIG.checkpoints_dir
gdrive_reports = CONFIG.reports_dir

if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(gdrive_dest, exist_ok=True)
    os.makedirs(gdrive_reports, exist_ok=True)
    for p in saved_ckpt_paths:
        shutil.copy(p, os.path.join(gdrive_dest, os.path.basename(p)))
    shutil.copy(preprocessor_save_path, os.path.join(gdrive_dest, "preprocessor.pkl"))
    shutil.copy(config_save_path, os.path.join(gdrive_dest, "experiment_config.json"))
    shutil.copy(metrics_file, os.path.join(gdrive_reports, "centralized_e1_metrics.json"))
    for fig_file in ["centralized_learning_curves.png", "centralized_confusion_matrix.png", "centralized_per_class_metrics.png", "centralized_minority_recall.png"]:
        src_fig = os.path.join("reports/figures", fig_file)
        if os.path.exists(src_fig):
            shutil.copy(src_fig, os.path.join(gdrive_reports, fig_file))
    print(f"\n🎉 All Top-{CONFIG.save_top_k} checkpoints, config, plots & reports permanently archived to Google Drive: {gdrive_dest}")
else:
    print("Google Drive not mounted. Results saved in local Colab session.")
